In [8]:
from __future__ import print_function
import numpy as np
from time import time # for comparing runing time
d, N = 1000, 10000 # dimension, number of training points
X = np.random.randn(N, d) # N d-dimensional points
z = np.random.randn(d)

In [12]:
# naively compute square distance between two vector
def dist_pp(z, x):
    d = z - x.reshape(z.shape) # force x and z to have the same dims
    return np.sum(d*d)
# from one point to each point in a set, naive
def dist_ps_naive(z, X):
    N = X.shape[0]
    res = np.zeros((1, N))
    for i in range(N):
        res[0][i] = dist_pp(z, X[i])
    return res
# from one point to each point in a set, fast
def dist_ps_fast(z, X):
    X2 = np.sum(X*X, 1) # square of l2 norm of each ROW of X
    z2 = np.sum(z*z) # square of l2 norm of z
    return X2 + z2 - 2*X.dot(z) # z2 can be ignored
t1 = time()
D1 = dist_ps_naive(z, X)

t2 = time()
print('slow point2set running time', t2 - t1, 's')

slow point2set running time 0.13381576538085938 s


In [14]:
import numpy as np
from time import time

M= 100
Z = np.random.randn(M, d)
# from each point in one set to each point in another set, half fast
def dist_ss_0(Z, X):
    M = Z.shape[0]
    N = X.shape[0]
    res = np.zeros((M, N))
    for i in range(M):
        res[i] = dist_ps_fast(Z[i], X)
    return res
# from each point in one set to each point in another set, fast
def dist_ss_fast(Z, X):
    X2 = np.sum(X*X, 1) # square of l2 norm of each ROW of X
    Z2 = np.sum(Z*Z, 1) # square of l2 norm of each ROW of Z
    return Z2.reshape(-1, 1) + X2.reshape(1, -1) - 2*Z.dot(X.T)
t1 = time()
D3 = dist_ss_0(Z, X)
naive_duration = time() - t1
print('slow set2set running time', naive_duration, 's')
t1 = time()
D4 = dist_ss_fast(Z, X)
fast_duration = time() - t1
print('slow set2set running time', fast_duration, 's')

naive_duration/fast_duration

slow set2set running time 5.1506781578063965 s
slow set2set running time 0.08011198043823242 s


64.29348181920992

Có thể sử dụng hàm cdist (https:// goo.gl/ vYMnmM) trong scipy.spatial.distance hoặc pairwise_distances (https:// goo.gl/ QK6Zyi) trong
sklearn.metrics.pairwise.

In [17]:
import scipy.spatial.distance as dist
import sklearn.metrics.pairwise as pairwise

D5 = dist.cdist(Z, X, 'sqeuclidean')
D6 = pairwise.pairwise_distances(Z, X, metric='euclidean')**2

![Flower](img\9_2_Irish_flower_img.png)

```
Iris flower dataset (https://goo.gl/eUy83R) is a small dataset. This dataset includes information about three classes of Iris flowers (a type of orchid): Iris setosa, Iris virginica, and Iris versicolor. Each class has 50 flowers with data consisting of four features: sepal length, sepal width, petal length, and petal width. Figure 9.2 is an example illustration of the three flower classes. Note that the data points are not images but are feature vectors consisting of the four pieces of information mentioned above.
```

In [25]:
from __future__ import print_function
import numpy as np
from sklearn import neighbors, datasets
from sklearn.model_selection import train_test_split # for splitting data
from sklearn.metrics import accuracy_score # for evaluating results
from sklearn.datasets import load_iris

In [26]:

np.random.seed(7)
iris = datasets.load_iris()
iris_X = iris.data
iris_y = iris.target
print('Labels:', np.unique(iris_y))
# split train and test
X_train, X_test, y_train, y_test = train_test_split(iris_X, iris_y, test_size=130)
print('Train size:', X_train.shape[0], ', test size:', X_test.shape[0])

Labels: [0 1 2]
Train size: 20 , test size: 130


In [36]:
def knn(k = 1, weights = 'uniform'):
    model = neighbors.KNeighborsClassifier(n_neighbors = k, p = 2, weights = weights)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    print(f"Accuracy of {k}NN_{weights}: %.2f %%" %(100*accuracy_score(y_test, y_pred)))

for k in range(1, 3):
    knn(k)
    knn(k, weights = 'distance')
    print('---')

Accuracy of 1NN_uniform: 92.31 %
Accuracy of 1NN_distance: 92.31 %
---
Accuracy of 2NN_uniform: 91.54 %
Accuracy of 2NN_distance: 92.31 %
---


In [56]:
# calculate the Euclidean distance between two vectors
def euclidean_distance(row1, row2):
	distance = 0.0
	for i in range(len(row1)-1):
		distance += (row1[i] - row2[i])**2
	return (distance)**0.5 ## Square root of the sum of squared distances
data = iris.data
row0 = data[0]

def get_neighbors(train, test_row, num_neighbors):
	distances = list()
	for train_row in train:
		dist = euclidean_distance(test_row, train_row)
		distances.append((train_row, dist))
	distances.sort(key=lambda tup: tup[1])
	neighbors = list()
	for i in range(num_neighbors):
		neighbors.append(distances[i][0])
	return neighbors

def predict_classification(train, test_row, num_neighbors):
	neighbors = get_neighbors(train, test_row, num_neighbors)
	print(neighbors)
	output_values = [row[-1] for row in neighbors]
	print(output_values)
	prediction = max(set(output_values), key=output_values.count)
	return prediction


get_neighbors(data, data[0], 3)
predict_classification(data, data[0], 3)

[array([5.1, 3.5, 1.4, 0.2]), array([5.1, 3.5, 1.4, 0.3]), array([5. , 3.5, 1.3, 0.3])]
[np.float64(0.2), np.float64(0.3), np.float64(0.3)]


np.float64(0.3)